In [21]:
# ============================================================
# Youth Opportunity Index (YOI) — v2 Notebook (Equal 7-domain weights)
#
# What this version fixes/updates:
# 1) Works when run from /notebooks (auto-detects repo root + /data)
# 2) Uses your UPDATED data inventory (ACS processed + raw ACS 2024 + PLACES + CIBRS + services)
# 3) Builds a tract-level index with 7 domains, equal weights by default
# 4) If a domain has no usable indicators yet, it gets a neutral score of 0.5
#    (so domains remain equally weighted, but missing domains don't distort rankings)
#
# Outputs:
#   data/processed/yoi/yoi_v2_scores.csv
#   data/processed/yoi/yoi_v2_normalized.csv
#   data/processed/yoi/yoi_v2_metadata.csv
# ============================================================

from __future__ import annotations

from pathlib import Path
import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Paths (RUN FROM /notebooks or anywhere)
# ------------------------------------------------------------
CWD = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    raise FileNotFoundError("Could not find repo root (a parent directory containing /data).")

REPO_ROOT = find_repo_root(CWD)
DATA_DIR = REPO_ROOT / "data"

print("Notebook CWD:", CWD)
print("Repo root:", REPO_ROOT)
print("Data dir:", DATA_DIR)

# ------------------------------------------------------------
# 1) Helpers
# ------------------------------------------------------------

from __future__ import annotations
from pathlib import Path
import re
import numpy as np
import pandas as pd

SD_PREFIX = "06073"

def normalize_geoid11(x) -> str | float:
    """
    Returns an 11-digit tract GEOID string (keeps leading zeros).
    Handles:
      - '1400000US06073000100'
      - 6073000100  (10-digit numeric)
      - '06073000100'
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip()

    # remove trailing .0 from floats rendered as strings
    s = re.sub(r"\.0$", "", s)

    digits = re.sub(r"\D", "", s)
    if digits == "":
        return np.nan

    # If <= 11 digits, left-pad (preserves leading 0 for CA)
    if len(digits) <= 11:
        return digits.zfill(11)

    # If longer, tract GEOID is typically the LAST 11 digits
    return digits[-11:]


def guess_geoid_col(df: pd.DataFrame) -> str | None:
    """Pick the most likely GEOID column."""
    preferred = [
        "tract_geoid", "GEOID", "geoid", "GEO_ID", "geo_id",
        "LocationName", "locationname", "id", "ID"
    ]
    cols_lc = {c.lower(): c for c in df.columns}
    for p in preferred:
        if p.lower() in cols_lc:
            return cols_lc[p.lower()]

    # fallback: score columns by how many values can be normalized to 11 digits
    best_col, best_score = None, -1
    sample = df.head(200)
    for c in df.columns:
        vals = sample[c].apply(normalize_geoid11)
        score = vals.notna().mean()
        if score > best_score:
            best_score = score
            best_col = c
    return best_col if best_score > 0 else None


def ensure_tract_geoid(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    geoid_col = guess_geoid_col(df)
    if geoid_col is None:
        raise ValueError("Could not find a GEOID-like column in this table.")
    df["tract_geoid"] = df[geoid_col].apply(normalize_geoid11)

    # enforce string dtype + keep only valid 11-digit values
    df["tract_geoid"] = df["tract_geoid"].astype("string")
    df.loc[df["tract_geoid"].str.len() != 11, "tract_geoid"] = pd.NA
    return df


def filter_sd_county(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.dropna(subset=["tract_geoid"])
    df = df[df["tract_geoid"].astype(str).str.startswith(SD_PREFIX)].copy()
    return df


def merge_on_tract(left: pd.DataFrame, right: pd.DataFrame, how="left") -> pd.DataFrame:
    if right is None or len(right) == 0:
        return left
    left = left.copy()
    right = right.copy()

    left["tract_geoid"]  = left["tract_geoid"].astype("string")
    right["tract_geoid"] = right["tract_geoid"].astype("string")

    dupes = (set(left.columns) & set(right.columns)) - {"tract_geoid"}
    right = right.drop(columns=list(dupes), errors="ignore")
    return left.merge(right, on="tract_geoid", how=how)

def safe_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def pick_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    return None

def pct_rank_01(s: pd.Series) -> pd.Series:
    """Percentile rank in [0,1]; NaNs preserved."""
    return s.rank(pct=True, method="average")

def normalize_indicator(series: pd.Series, higher_is_better: bool) -> pd.Series:
    r = pct_rank_01(series)
    return r if higher_is_better else (1.0 - r)

def load_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if path.exists():
        df = pd.read_csv(path)
        print("Loaded:", path, "shape:", df.shape)
        return df
    return None

def coerce_geoid11_str(s: pd.Series) -> pd.Series:
    """Force GEOID to 11-digit string, NaN if not valid."""
    out = s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
    out = out.str.replace(r"\D", "", regex=True)  # keep digits only
    out = out.where(out.str.len() == 11, np.nan)
    return out

# ------------------------------------------------------------
# 2) Load base tract tables you already have (processed)
#    (These are often already filtered to SD tracts.)
# ------------------------------------------------------------
base_candidates = [
    DATA_DIR / "processed" / "acs5" / "2023" / "tract_opportunity_desert_metrics_2023.csv",
    DATA_DIR / "processed" / "acs"  / "acs_tract_indicators.csv",
    DATA_DIR / "processed" / "acs5" / "2023" / "acs_tract_demographics_2023.csv",
    DATA_DIR / "processed" / "acs5" / "2023" / "tract_with_equity_analysis.csv",
]

dfs = []
for p in base_candidates:
    if not p.exists():
        continue
    d = pd.read_csv(p)
    d = ensure_tract_geoid(d)
    d = filter_sd_county(d)

    print(f"{p.name}: rows after SD filter = {len(d)}")
    if len(d) > 0:
        dfs.append(d)

if len(dfs) == 0:
    raise FileNotFoundError("All base tables became empty after GEOID parsing/filtering. Check GEOID handling.")

# choose base with most rows (then most columns)
dfs = sorted(dfs, key=lambda x: (x.shape[0], x.shape[1]), reverse=True)
tracts = dfs[0].copy()
for d in dfs[1:]:
    tracts = merge_on_tract(tracts, d, how="left")

print("Merged tract base shape:", tracts.shape)
print("Example GEOIDs:", tracts["tract_geoid"].dropna().head(5).tolist())

# ------------------------------------------------------------
# 3) Services → counts + services_per_10k (robust merge)
# ------------------------------------------------------------
services_path = DATA_DIR / "processed" / "services" / "services_master.csv"
services = load_csv_if_exists(services_path)
services = ensure_tract_geoid(services)
services = filter_sd_county(services)

svc_counts = (
    services.groupby("tract_geoid")
    .size()
    .reset_index(name="service_count")
)

# Merge with suffix to avoid collisions
tracts = tracts.merge(svc_counts, on="tract_geoid", how="left", suffixes=("", "_svc"))

# If merge created a suffixed column instead of service_count, recover it
if "service_count" not in tracts.columns:
    if "service_count_svc" in tracts.columns:
        tracts["service_count"] = tracts["service_count_svc"]
    else:
        tracts["service_count"] = 0

tracts["service_count"] = tracts["service_count"].fillna(0).astype(int)

# population for rates
pop_col = pick_col(tracts, ["total_population", "population", "pop_total", "B01003_001E", "total_pop"])

if pop_col is not None:
    tracts[pop_col] = safe_numeric(tracts[pop_col])
    tracts["services_per_10k"] = np.where(
        tracts[pop_col] > 0,
        tracts["service_count"] / tracts[pop_col] * 10000.0,
        np.nan
    )
else:
    tracts["services_per_10k"] = np.nan
    print("WARNING: No population column found → services_per_10k cannot be computed.")

# ------------------------------------------------------------
# 4) PLACES (health outcomes) → wide measures per tract
# ------------------------------------------------------------
places_path = DATA_DIR / "raw" / "benchmarks" / "PLACES__Local_Data_for_Better_Health,_Census_Tract_Data,_2025_release_20260308.csv"
places = load_csv_if_exists(places_path)

if places is not None:
    places = ensure_tract_geoid(places)

    # Filter to SD County if CountyFIPS exists
    countyfips_col = pick_col(places, ["CountyFIPS", "countyfips"])
    if countyfips_col:
        places[countyfips_col] = places[countyfips_col].astype(str).str.zfill(5)
        places = places[places[countyfips_col] == "06073"].copy()
    else:
        # fallback: filter by tract prefix
        places = filter_sd_county(places)

    year_col = pick_col(places, ["Year", "year"])
    if year_col:
        latest_year = safe_numeric(places[year_col]).max()
        places = places[safe_numeric(places[year_col]) == latest_year].copy()
        print("PLACES: using latest year =", int(latest_year))

    meas_col = pick_col(places, ["MeasureId", "measureid", "MeasureID"])
    val_col  = pick_col(places, ["Data_Value", "data_value", "DataValue"])
    if meas_col and val_col:
        places[val_col] = safe_numeric(places[val_col])

        keep_measures = [
            # Health access / need
            "ACCESS2",     # uninsured
            "MHLTH",       # frequent mental distress
            "PHLTH",       # poor physical health
            "DEPRESSION",  # depression
            "OBESITY",
            "DIABETES",
            # Mobility / constraints
            "LACKTRPT",    # lack of transportation
            # Safety / environment-ish
            "FOODINSECU",  # food insecurity proxy
            "HOUSINSECU",  # housing insecurity proxy
        ]
        available = set(places[meas_col].dropna().astype(str).unique())
        keep = [m for m in keep_measures if m in available]

        if len(keep) == 0:
            # fallback: top 8 measures by frequency
            keep = places[meas_col].astype(str).value_counts().head(8).index.tolist()
            print("PLACES: default measures not found; using most frequent:", keep)
        else:
            print("PLACES: keeping measures:", keep)

        places_sub = places[places[meas_col].astype(str).isin(keep)].copy()
        wide = (
            places_sub.pivot_table(index="tract_geoid", columns=meas_col, values=val_col, aggfunc="mean")
            .reset_index()
        )
        wide.columns = ["tract_geoid"] + [f"places_{c}" for c in wide.columns if c != "tract_geoid"]

        tracts = merge_on_tract(tracts, wide, how="left")
    else:
        print("WARNING: PLACES file missing MeasureId/Data_Value columns; skipping.")

# ------------------------------------------------------------
# 5) Crime (CIBRS detailed) → incidents per tract + rate per 1k (robust)
# ------------------------------------------------------------
cibrs_path = DATA_DIR / "raw" / "benchmarks" / "CIBRS_Group_A_Detailed_Report_Data_20260308.csv"

# Ensure columns exist even if we skip crime
if "crime_groupA_incidents" not in tracts.columns:
    tracts["crime_groupA_incidents"] = np.nan
if "crime_rate_per_1k" not in tracts.columns:
    tracts["crime_rate_per_1k"] = np.nan

if not cibrs_path.exists():
    print("NOTE: CIBRS not found; skipping crime.")
else:
    print("Found CIBRS:", cibrs_path)

    cols0 = pd.read_csv(cibrs_path, nrows=0).columns.tolist()

    # Find key columns
    tract_col = next((c for c in cols0 if c.lower().strip() == "census tract"), None)
    inc_uid_col = next((c for c in cols0 if "incident uid" in c.lower()), None)
    unique_inc_col = next((c for c in cols0 if "unique incidents" in c.lower()), None)
    pop_ct_col = next((c for c in cols0 if "census tract population" in c.lower()), None)

    if tract_col is None:
        print("WARNING: No 'Census Tract' column found in CIBRS; skipping crime.")
    else:
        usecols = [c for c in [tract_col, inc_uid_col, unique_inc_col, pop_ct_col] if c is not None]
        cibrs = pd.read_csv(cibrs_path, usecols=usecols)

        cibrs["tract_geoid"] = cibrs[tract_col].apply(extract_geoid11)
        cibrs = filter_sd_county(cibrs)

        # Aggregate incidents by tract
        if inc_uid_col:
            crime_counts = (
                cibrs.groupby("tract_geoid")[inc_uid_col]
                .nunique()
                .reset_index(name="crime_groupA_incidents")
            )
        elif unique_inc_col:
            cibrs[unique_inc_col] = safe_numeric(cibrs[unique_inc_col]).fillna(0)
            crime_counts = (
                cibrs.groupby("tract_geoid")[unique_inc_col]
                .sum()
                .reset_index(name="crime_groupA_incidents")
            )
        else:
            crime_counts = (
                cibrs.groupby("tract_geoid")
                .size()
                .reset_index(name="crime_groupA_incidents")
            )

        # Merge with suffix to avoid collisions
        # Ensure tract_geoid types match (string) on BOTH tables
        tracts["tract_geoid"] = coerce_geoid11_str(tracts["tract_geoid"])
        crime_counts["tract_geoid"] = coerce_geoid11_str(crime_counts["tract_geoid"])

        tracts = tracts.merge(crime_counts, on="tract_geoid", how="left", suffixes=("", "_crime"))

        # Recover correct column name if collision happened
        if "crime_groupA_incidents" not in tracts.columns and "crime_groupA_incidents_crime" in tracts.columns:
            tracts["crime_groupA_incidents"] = tracts["crime_groupA_incidents_crime"]

        tracts["crime_groupA_incidents"] = safe_numeric(tracts["crime_groupA_incidents"]).fillna(0)

        # Denominator: prefer CIBRS tract population if present, else your tract population
        denom = None
        if pop_ct_col:
            pop_ct = (
                cibrs.groupby("tract_geoid")[pop_ct_col]
                .first()
                .reset_index()
                .rename(columns={pop_ct_col: "cibrs_tract_pop"})
            )

            # --- FIX: force tract_geoid to 11-digit string on BOTH sides ---
            pop_ct["tract_geoid"] = coerce_geoid11_str(pop_ct["tract_geoid"])
            tracts["tract_geoid"] = coerce_geoid11_str(tracts["tract_geoid"])

            pop_ct["cibrs_tract_pop"] = safe_numeric(pop_ct["cibrs_tract_pop"])

            tracts = tracts.merge(pop_ct, on="tract_geoid", how="left", suffixes=("", "_pop"))

            if "cibrs_tract_pop" in tracts.columns:
                denom = "cibrs_tract_pop"

        if denom is None:
            denom = pop_col if (pop_col is not None and pop_col in tracts.columns) else None

        if denom:
            tracts[denom] = safe_numeric(tracts[denom])
            tracts["crime_rate_per_1k"] = np.where(
                tracts[denom] > 0,
                tracts["crime_groupA_incidents"] / tracts[denom] * 1000.0,
                np.nan
            )
        else:
            print("WARNING: No population column available to compute crime_rate_per_1k.")

# ------------------------------------------------------------
# 6) Pull additional mobility/education/disability from raw ACS 2024
#    (These are already in your inventory. We'll extract a few key indicators.)
# ------------------------------------------------------------
def load_census_download(data_csv: Path) -> pd.DataFrame | None:
    if not data_csv.exists():
        return None
    df = pd.read_csv(data_csv)
    df = ensure_tract_geoid(df)
    df = filter_sd_county(df)
    return df

# 6A) Commute time (mean travel time to work) from B08303
b08303 = load_census_download(DATA_DIR / "raw" / "acs" / "2024" / "ACSDT5Y2024.B08303-Data.csv")
if b08303 is not None:
    # Try common mean travel time column
    # Often B08303_001E = Mean travel time to work (minutes)
    cand = [c for c in b08303.columns if re.fullmatch(r".*B08303_001E$", c)] + ["B08303_001E"]
    col = pick_col(b08303, cand)
    if col is None:
        # fallback: first estimate-like column
        est_cols = [c for c in b08303.columns if c.endswith("E")]
        col = est_cols[0] if est_cols else None
    if col:
        temp = b08303[["tract_geoid", col]].rename(columns={col: "mean_commute_minutes"})
        temp["mean_commute_minutes"] = safe_numeric(temp["mean_commute_minutes"])
        tracts = merge_on_tract(tracts, temp, how="left")

# 6B) Education attainment from S1501 (HS+ and BA+ percent)
s1501 = load_census_download(DATA_DIR / "raw" / "acs" / "2024" / "ACSST5Y2024.S1501-Data.csv")
if s1501 is not None:
    # We don't assume exact S1501 column codes — instead try keyword-ish fallbacks by common codes first
    # If these don't exist, the notebook will still run; you'll just see these as missing.
    # Common S1501 percent columns (these vary by release):
    likely = {
        "hs_plus_pct":  ["S1501_C02_015E", "S1501_C02_014E", "S1501_C02_012E"],
        "ba_plus_pct":  ["S1501_C02_016E", "S1501_C02_015E", "S1501_C02_013E"],
    }
    out = s1501[["tract_geoid"]].copy()
    for outcol, cands in likely.items():
        found = pick_col(s1501, cands)
        if found:
            out[outcol] = safe_numeric(s1501[found])
    tracts = merge_on_tract(tracts, out, how="left")

# 6C) Disability % from S1810
s1810 = load_census_download(DATA_DIR / "raw" / "acs" / "2024" / "ACSST5Y2024.S1810-Data.csv")
if s1810 is not None:
    # Common disability percent column candidates (varies)
    # We'll try a few likely ones; if not found, you still keep PLACES DISABILITY as a proxy if you add it.
    likely_dis = ["S1810_C02_001E", "S1810_C02_002E", "S1810_C02_003E"]
    col = pick_col(s1810, likely_dis)
    if col:
        temp = s1810[["tract_geoid", col]].rename(columns={col: "disability_pct"})
        temp["disability_pct"] = safe_numeric(temp["disability_pct"])
        tracts = merge_on_tract(tracts, temp, how="left")

# 6D) Broadband subscription proxy from K202801 (if tract-level rows exist)
k202801 = load_census_download(DATA_DIR / "raw" / "acs" / "2024" / "ACSSE2024.K202801-Data.csv")
if k202801 is not None:
    # try to find any column with "broadband" in the name; if not, try common percent estimate column
    bb_cols = [c for c in k202801.columns if "broadband" in c.lower() and c.endswith("E")]
    col = bb_cols[0] if bb_cols else None
    if col is None:
        est_cols = [c for c in k202801.columns if c.endswith("E")]
        col = est_cols[0] if est_cols else None
    if col:
        temp = k202801[["tract_geoid", col]].rename(columns={col: "broadband_proxy"})
        temp["broadband_proxy"] = safe_numeric(temp["broadband_proxy"])
        tracts = merge_on_tract(tracts, temp, how="left")

# ------------------------------------------------------------
# 7) Define the 7 domains + indicator specification
#    Equal domain weights (1/7 each) by default.
# ------------------------------------------------------------
DOMAINS = [
    "economic",
    "education",
    "health",
    "housing",
    "safety_env",
    "mobility_connectivity",
    "youth_supports",
]
DEFAULT_DOMAIN_WEIGHTS = {d: 1/len(DOMAINS) for d in DOMAINS}

# We’ll map indicators by *finding the column that exists* in your merged tracts table.
# If not found, it's simply skipped; domains with no indicators become neutral 0.5.
INDICATORS = [
    # --- Economic ---
    dict(name="poverty_rate", domain="economic", higher_is_better=False,
         col_candidates=["poverty_rate_pct", "poverty_pct", "pct_poverty", "S1701_C02_001E"]),
    dict(name="median_household_income", domain="economic", higher_is_better=True,
         col_candidates=["median_household_income", "med_hh_income", "B19013_001E", "income_median", "K201902"]),
    dict(name="snap_rate", domain="economic", higher_is_better=False,
         col_candidates=["snap_rate", "foodstamp_pct", "food_stamps_pct", "S2201_C02_013E", "FOODSTAMP", "places_FOODSTAMP"]),

    # --- Education ---
    dict(name="hs_plus_pct", domain="education", higher_is_better=True,
         col_candidates=["hs_plus_pct", "hs_or_higher_pct", "hs_or_higher", "S1501"]),
    dict(name="ba_plus_pct", domain="education", higher_is_better=True,
         col_candidates=["ba_plus_pct", "ba_or_higher_pct", "ba_or_higher", "S1501"]),
    
    # --- Health ---
    dict(name="uninsured_rate", domain="health", higher_is_better=False,
         col_candidates=["uninsured_rate", "pct_uninsured", "places_ACCESS2"]),
    dict(name="mental_distress", domain="health", higher_is_better=False,
         col_candidates=["mental_distress", "places_MHLTH"]),
    dict(name="poor_physical_health", domain="health", higher_is_better=False,
         col_candidates=["poor_physical_health", "places_PHLTH"]),
    dict(name="depression", domain="health", higher_is_better=False,
         col_candidates=["places_DEPRESSION"]),
    dict(name="obesity", domain="health", higher_is_better=False,
         col_candidates=["places_OBESITY"]),
    dict(name="diabetes", domain="health", higher_is_better=False,
         col_candidates=["places_DIABETES"]),
    dict(name="disability_pct", domain="health", higher_is_better=False,
         col_candidates=["disability_pct", "places_DISABILITY"]),

    # --- Housing (you may still be missing tract-level rent burden/overcrowding) ---
    dict(name="housing_insecurity", domain="housing", higher_is_better=False,
         col_candidates=["places_HOUSINSECU"]),
    dict(name="food_insecurity", domain="housing", higher_is_better=False,
         col_candidates=["places_FOODINSECU"]),

    # --- Safety & environment ---
    dict(name="crime_rate_per_1k", domain="safety_env", higher_is_better=False,
         col_candidates=["crime_rate_per_1k"]),
    
    # --- Mobility & connectivity ---
    dict(name="no_vehicle_rate", domain="mobility_connectivity", higher_is_better=False,
         col_candidates=["no_vehicle_rate", "pct_no_vehicle", "B08201_002E"]),
    dict(name="mean_commute_minutes", domain="mobility_connectivity", higher_is_better=False,
         col_candidates=["mean_commute_minutes", "mean_commute_time", "avg_commute_time"]),
    dict(name="lack_transport", domain="mobility_connectivity", higher_is_better=False,
         col_candidates=["places_LACKTRPT"]),
    dict(name="broadband", domain="mobility_connectivity", higher_is_better=True,
         col_candidates=["broadband_rate", "broadband_proxy", "internet_subscription", "pct_broadband"]),

    # --- Youth supports / wraparound ---
    dict(name="service_count", domain="youth_supports", higher_is_better=True,
         col_candidates=["service_count"]),
    dict(name="services_per_10k", domain="youth_supports", higher_is_better=True,
         col_candidates=["services_per_10k"]),
]

def resolve_indicator_columns(df: pd.DataFrame, indicators: list[dict]) -> pd.DataFrame:
    rows = []
    for spec in indicators:
        found = None
        for cand in spec["col_candidates"]:
            # exact match
            if cand in df.columns:
                found = cand
                break
            # fuzzy: if cand looks like a prefix, match any column containing it
            matches = [c for c in df.columns if cand.lower() in c.lower()]
            if len(matches) == 1:
                found = matches[0]
                break
        if found is not None:
            rows.append({
                "indicator": spec["name"],
                "domain": spec["domain"],
                "col": found,
                "higher_is_better": spec["higher_is_better"],
            })
    meta = pd.DataFrame(rows).sort_values(["domain", "indicator"])
    return meta

meta = resolve_indicator_columns(tracts, INDICATORS)
print("Indicators included:", len(meta))
missing_domains = sorted(set(DOMAINS) - set(meta["domain"].unique()))
print("Domains with 0 indicators (will be neutral 0.5):", missing_domains)
display(meta)

# ------------------------------------------------------------
# 8) Compute normalized indicators + domain scores + overall YOI
# ------------------------------------------------------------
# normalized indicators df
norm = tracts[["tract_geoid"]].copy()

for _, r in meta.iterrows():
    col = r["col"]
    x = safe_numeric(tracts[col])
    norm[f"norm_{r['indicator']}"] = normalize_indicator(x, bool(r["higher_is_better"]))

# domain scores
scores = tracts[["tract_geoid"]].copy()

domain_summary_rows = []
domain_scores = {}

for d in DOMAINS:
    inds = meta[meta["domain"] == d]["indicator"].tolist()
    if len(inds) == 0:
        # neutral domain if no indicators available yet
        scores[f"{d}_score"] = 0.5
        scores[f"{d}_coverage"] = 0.0
        domain_summary_rows.append({"domain": d, "num_indicators_used": 0, "avg_domain_coverage": 0.0})
        continue

    cols = [f"norm_{i}" for i in inds]
    mat = norm[cols]

    scores[f"{d}_coverage"] = mat.notna().mean(axis=1)  # fraction present for that tract
    scores[f"{d}_score"] = mat.mean(axis=1, skipna=True)

    domain_summary_rows.append({
        "domain": d,
        "num_indicators_used": len(inds),
        "avg_domain_coverage": float(scores[f"{d}_coverage"].mean())
    })

domain_summary = pd.DataFrame(domain_summary_rows)
display(domain_summary)

# overall YOI with equal domain weights
weights = DEFAULT_DOMAIN_WEIGHTS.copy()

# If you want to experiment later:
# weights = {"economic": 1/7, "education": 1/7, ... }  # must sum to 1

# Ensure weights sum to 1 (normalize)
s = sum(weights.values())
weights = {k: v/s for k, v in weights.items()}

scores["yoi_raw_0_1"] = 0.0
for d, w in weights.items():
    scores["yoi_raw_0_1"] += scores[f"{d}_score"] * w

scores["yoi_0_100"] = scores["yoi_raw_0_1"] * 100.0

# Attach a few raw context fields (optional)
for c in ["service_count", "services_per_10k", pop_col]:
    if c and c in tracts.columns:
        scores[c] = tracts[c]

# ------------------------------------------------------------
# 9) Save outputs
# ------------------------------------------------------------
out_dir = DATA_DIR / "processed" / "yoi"
out_dir.mkdir(parents=True, exist_ok=True)

scores_path = out_dir / "yoi_v2_scores.csv"
norm_path   = out_dir / "yoi_v2_normalized.csv"
meta_path   = out_dir / "yoi_v2_metadata.csv"

scores.to_csv(scores_path, index=False)
norm.to_csv(norm_path, index=False)
meta.to_csv(meta_path, index=False)

print("\nSaved:")
print(" -", scores_path)
print(" -", norm_path)
print(" -", meta_path)

# Quick sanity check
display(scores.describe(include="all"))

Notebook CWD: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/notebooks
Repo root: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts
Data dir: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data
tract_opportunity_desert_metrics_2023.csv: rows after SD filter = 912
acs_tract_indicators.csv: rows after SD filter = 737
acs_tract_demographics_2023.csv: rows after SD filter = 737
tract_with_equity_analysis.csv: rows after SD filter = 877
Merged tract base shape: (1452, 37)
Example GEOIDs: ['06073008331', '06073008331', '06073008331', '06073008331', '06073008331']
Loaded: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/processed/services/services_master.csv shape: (5905, 18)
Loaded: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/raw/benchmarks/PLACES__Local_Data_for_Better_Health,_Census_Tract_Data,_2025_release_20260308.csv shape: (29397, 24)
PLACES: using latest year = 2023
PLACES: keep

,indicator,domain,col,higher_is_better
0,poverty_rate,economic,poverty_rate_pct,False
4,depression,health,places_DEPRESSION,False
6,diabetes,health,places_DIABETES,False
7,disability_pct,health,disability_pct,False
2,mental_distress,health,places_MHLTH,False
5,obesity,health,places_OBESITY,False
3,poor_physical_health,health,places_PHLTH,False
1,uninsured_rate,health,places_ACCESS2,False
9,food_insecurity,housing,places_FOODINSECU,False
8,housing_insecurity,housing,places_HOUSINSECU,False


,domain,num_indicators_used,avg_domain_coverage
0,economic,1,0.996556
1,education,0,0.000000
2,health,7,0.998819
3,housing,2,0.998623
4,safety_env,1,0.000000
5,mobility_connectivity,2,0.999311
6,youth_supports,2,0.999311



Saved:
 - /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/processed/yoi/yoi_v2_scores.csv
 - /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/processed/yoi/yoi_v2_normalized.csv
 - /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/processed/yoi/yoi_v2_metadata.csv


,tract_geoid,economic_coverage,economic_score,education_score,education_coverage,health_coverage,health_score,housing_coverage,housing_score,safety_env_coverage,safety_env_score,mobility_connectivity_coverage,mobility_connectivity_score,youth_supports_coverage,youth_supports_score,yoi_raw_0_1,yoi_0_100,service_count,services_per_10k,total_population
count,1452,1452.000000,1447.000000,1452.0,1452.0,1452.000000,1452.000000,1452.000000,1450.000000,1452.0,0.0,1452.000000,1452.000000,1452.000000,1452.000000,0.0,0.0,1452.000000,1450.000000,1452.000000
unique,737,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,06073008504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.996556,0.499654,0.5,0.0,0.998819,0.500243,0.998623,0.499655,0.0,NaN,0.999311,0.499999,0.999311,0.500246,NaN,NaN,13.044766,28.967962,4559.610882
std,NaN,0.058601,0.288696,0.0,0.0,0.031801,0.222833,0.037101,0.287950,0.0,NaN,0.018550,0.240055,0.018550,0.281718,NaN,NaN,17.153342,44.918473,1791.655623
min,NaN,0.000000,0.000000,0.5,0.0,0.142857,0.017437,0.000000,0.000000,0.0,NaN,0.500000,0.001033,0.500000,0.018263,NaN,NaN,0.000000,0.000000,0.000000
25%,NaN,1.000000,0.234969,0.5,0.0,1.000000,0.306144,1.000000,0.252026,0.0,NaN,1.000000,0.293311,1.000000,0.259278,NaN,NaN,4.000000,9.615385,3554.000000
50%,NaN,1.000000,0.497236,0.5,0.0,1.000000,0.501384,1.000000,0.517241,0.0,NaN,1.000000,0.505412,1.000000,0.488097,NaN,NaN,8.000000,19.555412,4436.000000
75%,NaN,1.000000,0.751209,0.5,0.0,1.000000,0.682074,1.000000,0.755000,0.0,NaN,1.000000,0.667545,1.000000,0.751166,NaN,NaN,15.250000,37.313433,5594.250000
